In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from gbmcollaboration.definitions import AVG_NUM_DAYS_PER_YEAR
from gbmcollaboration import AllDataCenters, UKERDataCenter
from gbmcollaboration.utils import (
    get_datasets,
    s3_ls,
)


def load_clinical():
    def _get_clin_key(source_dict,key):
        df = (pd.concat(
            {k: v["clinical"][key] 
            for k,v in source_dict.items()})
            .reset_index()
            .rename(columns={'level_0':'center'}))
        return df

    source_dict = AllDataCenters.load_tabular(exclude_idh1_mut=True)
    df_clin_processed = _get_clin_key(source_dict,"processed gbm clinical")

    # add days unit and signed format (-1 for censored)
    df_clin_processed['os_days'] = df_clin_processed['os_years'] * AVG_NUM_DAYS_PER_YEAR
    df_clin_processed['pfs_days'] = df_clin_processed['pfs_years'] * AVG_NUM_DAYS_PER_YEAR
    df_clin_processed['os_days_signed'] = df_clin_processed['os_days'] * (df_clin_processed['os_event'].apply(lambda e: 1 if e == 1 else -1))
    df_clin_processed['pfs_days_signed'] = df_clin_processed['pfs_days'] * (df_clin_processed['pfs_event'].apply(lambda e: 1 if e == 1 else -1))

    return df_clin_processed

def get_best_row_per_group(df: pd.DataFrame, grouping_cols: list, rank_by_col: str, mode='min') -> pd.DataFrame:
    """
    For each group, finds the single row corresponding to the minimum value
    of a specified ranking column.

    Args:
        df: The input DataFrame.
        grouping_cols: A list of column names to group by.
        rank_by_col: The column name to use for finding the minimum value.

    Returns:
        A DataFrame containing only the best row from each group.
    """
    print(f"Original number of rows: {len(df)}")
    
    # 1. Group by the desired columns and find the index of the minimum value
    #    in the rank_by_col for each group.
    if mode=='min':
        idx = df.groupby(grouping_cols)[rank_by_col].idxmin()
    elif mode=='max':
        idx = df.groupby(grouping_cols)[rank_by_col].idxmin()

    # 2. Use these indices to select the full rows from the original DataFrame.
    result_df = df.loc[idx]
    
    print(f"Number of rows after selecting the {mode} per group: {len(result_df)}")
    
    return result_df


def calculate_patient_neoantigen_scores(
    df_mhc: pd.DataFrame,
    percentile_threshold: float = 1.0,
    top_n: int = 10,
    patient_col: str = 'Subject Id',
    peptide_col: str = 'peptide',
    score_col: str = 'presentation_percentile'
) -> pd.DataFrame:
    """
    Combines multiple strategies to calculate a set of neoantigen scores per patient.

    This function takes raw, multi-allele peptide presentation data and returns a
    single DataFrame with one row per patient and columns for each scoring metric.

    The process is as follows:
    1.  For each unique peptide within a patient, it finds the single best
        presentation (row with the minimum 'score_col' value).
    2.  It then calculates three distinct scores based on this best-hit data:
        - burden_count: The number of unique peptides with a score below the threshold.
        - weighted_burden: The sum of transformed scores for peptides below the threshold.
        - top_n_avg_score: The average score of the top N best peptides.

    Args:
        df_mhc: The raw DataFrame from MHCflurry (or similar), with columns for
                patient, peptide, allele, and presentation scores.
        percentile_threshold: The score threshold below which a peptide is
                              considered a high-quality neoantigen.
        top_n: The number of top peptides to consider for the 'top_n_avg_score'.
        patient_col: The name of the patient identifier column.
        peptide_col: The name of the peptide sequence column.
        score_col: The name of the presentation score/percentile column (lower is better).

    Returns:
        A pandas DataFrame with one row per patient and columns for each of the
        three calculated scores.
    """
    # --- Step 1: Pre-processing - Find the best presentation for each peptide per patient ---
    print(f"Original number of peptide-allele combinations: {len(df_mhc)}")
    
    # Find the index of the row with the minimum score for each patient-peptide group
    idx_of_best_presentation = df_mhc.groupby([patient_col, peptide_col])[score_col].idxmin()
    
    # Select these "best hit" rows to work with
    best_hits_df = df_mhc.loc[idx_of_best_presentation].copy()
    print(f"Number of unique peptides after collapsing alleles: {len(best_hits_df)}")

    # --- Step 2: Method 1 - Neoantigen Burden Count ---
    # Filter for high-quality peptides based on the threshold
    high_quality_peptides = best_hits_df[best_hits_df[score_col] < percentile_threshold]
    
    # Group by patient and count the number of unique peptides
    burden_scores = high_quality_peptides.groupby(patient_col)[peptide_col].nunique()
    burden_scores.name = 'burden_count'

    # --- Step 3: Method 2 - Weighted Burden Score ---
    # We use the same 'high_quality_peptides' DataFrame from the previous step
    # Transform score so higher is better (e.g., a 0.1 percentile -> 0.999)
    high_quality_peptides['immunogenicity_score'] = 1 - (high_quality_peptides[score_col] / 100.0)
    
    # Group by patient and sum these transformed scores
    weighted_scores = high_quality_peptides.groupby(patient_col)['immunogenicity_score'].sum()
    weighted_scores.name = 'weighted_burden'

    # --- Step 4: Method 3 - Average Score of Top N Peptides ---
    # For each patient, get the top N peptides based on the score
    # We use .groupby().apply() with nsmallest for this
    top_n_peptides = best_hits_df.groupby(patient_col).apply(
        lambda grp: grp.nsmallest(top_n, score_col)
    ).reset_index(drop=True)
    
    # Now, average the score for these top hits for each patient
    top_n_scores = top_n_peptides.groupby(patient_col)[score_col].mean()
    top_n_scores.name = 'top_n_avg_score'

    # --- Step 5: Combine all scores into a single DataFrame ---
    # Get a unique list of all patients from the original data
    all_patients = df_mhc[[patient_col]].drop_duplicates().set_index(patient_col)
    
    # Combine the calculated scores. Using .join is robust to missing patients.
    final_scores_df = all_patients.join([burden_scores, weighted_scores, top_n_scores])
    
    # Fill NaN values with 0. This is crucial for patients who had no peptides
    # passing the threshold, ensuring they get a score of 0 instead of NaN.
    final_scores_df.fillna(0, inplace=True)
    
    return final_scores_df.reset_index()

## parameters

In [ ]:
RESULTS_DIR = Path('/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/results')
p_mhc = RESULTS_DIR / 'WES/hlatyping/results/MHCflurry/MHCflurry_predictions.parquet'

In [ ]:
# datasets = get_datasets()
# wes_s3 = datasets['MOSAIC - CHUV - GBM - Processed WES Data']
# list(s3_ls(wes_s3.bucket,wes_s3.prefix+'live',recursive=False))

## read

In [ ]:
UKERDataCenter.load_tabular?

In [ ]:
# - clinical:
#     - original clinical: raw clinical data that have been processed by mosaic pipeline
#     - key events: summary of the date clinical data processed by mosaic pipeline
#     - processed gbm clinical: further processing in this repository
#     - endpoints: outcome data processed in this repository

cols = [
    # general info
    'Site code','Subject Id','Date of cancer diagnosis',
    # survival info
    'os_days_signed','os_days','os_years','os_event','os_start_event',
    'pfs_days_signed','pfs_days','pfs_years','pfs_event','pfs_start_event',
    # predictive vars
    'Administrative gender',
    'Age (years)',
    'Performance status (ECOG) at cancer diagnosis',
    'MGMT promoter methylation'
]

# source_dict = AllDataCenters.load_tabular(exclude_idh1_mut=True)
# clinical_df = source_dict["clinical"]["processed gbm clinical"][cols].drop_duplicates() #'endpoints'
# # If OS < 0, the event occurred (status = 1). Otherwise, it's censored (status = 0).
# clinical_df['OS_status'] = np.where(clinical_df['OS'] < 0, 1, 0)
# clinical_df['OS_duration'] = clinical_df['OS'].abs()
# clinical_df.index.name  = 'patient_id'

# load clinical data
df_clin_processed = load_clinical()
#   remove treated by IT or ADC
df_clin_processed_no_dups = df_clin_processed[~df_clin_processed[['received_immunotherapy','received_antibody_drug_conjugate']].any(axis=1)]
#   remove duplicates
df_clin_processed_no_dups = df_clin_processed_no_dups[cols].drop_duplicates().set_index('Subject Id')


df_mhc = pd.read_parquet(p_mhc).rename(columns={'patient_id':'sample_id'})
df_mhc['Subject Id'] = df_mhc['sample_id'].str.replace('_S000021','').str[:-1]

# get row with min presentation percentile
GROUPING_COLUMNS = ['Subject Id', 'allele', 'peptide']
RANK_BY_COLUMN = 'presentation_percentile'
best_df_mhc = get_best_row_per_group(
    df=df_mhc,
    grouping_cols=GROUPING_COLUMNS,
    rank_by_col=RANK_BY_COLUMN
)

patient_scores = calculate_patient_neoantigen_scores(
    best_df_mhc,
    percentile_threshold=0.5, # Peptides with percentile < 1.0 are "good"
    top_n=50                   # Consider top 50 peptides for the third score
)

df_analysis = pd.merge(df_clin_processed_no_dups, patient_scores, on='Subject Id', how='left').dropna()

In [ ]:
np.unique(df_analysis['Subject Id'])

In [ ]:
# load snRNA
# ad_chuv = CHUVDataCenter.load_singlecell()
ad_uker = UKERDataCenter.load_singlecell()
# ad = sc.concat((ad_chuv,ad_uker))
ad = ad_uker

# save memory
# %xdel ad_chuv
# %xdel ad_uker
# ad.raw = None

ad.obs['Subject Id']=ad.obs['orig.ident'].str[:-1]
ad.obs = ad.obs.join(df_clin_processed_no_dups,on='Subject Id')

In [ ]:
from gbmcollaboration.viz import calculate_proportions
CELLTYPE_KEY = 'celltype_level4_curated'
T_CELL_LABELS_IN_CELLTYPE_KEY = [
    'T_reg',
    'T_CD8_nos',
    'T_CD4_nos',
    'T_other',
    'T_NK_cycling',
    'T_nos',
    'T_IFN_activated',
    'B_nos',
    'Plasma_nos',
    'Plasmablast',
    #  'Doublet_TNK_Malignant',
    #  'Doublet_TNK_Mural',
    #  'Doublet_TNK_Myeloid',
    #  'Doublet_TNK_nos',
    #  'Doublet_T_Oligodendro',
]

BATCH_KEY = 'Subject Id'
CELLTYPE_KEY = 'celltype_level4_curated'
# 1. Calculate Malignant Proportions
# (Using your existing calculate_proportions function)
comp_df = calculate_proportions(ad.obs, BATCH_KEY, CELLTYPE_KEY)

t_cell_percentages = (comp_df[T_CELL_LABELS_IN_CELLTYPE_KEY].sum(1)*100).to_frame()
t_cell_percentages.columns = ['TIL_pct']
t_cell_percentages.hist(bins=50)

## analysis

In [ ]:
dict_ct_simple = {
    'malignant':['AC1-like',
    'AC2-like',
    'AC-like_IFN_response',
    'MES-like',
    'Malignant_MES2-like',
    'Malignant_cycling',
    'Malignant_other_1',
    'Malignant_OPC_invasive',],

    'malignant_AC-like':['AC1-like',
    'AC2-like',
    'AC-like_IFN_response'],

    'malignant_MES-like':
    ['MES-like',
    'Malignant_MES2-like',],

    'malignant_OPC-like':
    ['Malignant_OPC_invasive',],

    'DC':['cDC1_nos',
    'cDC2_cycling',
    'cDC2_nos',
    'cDC_CD207',
    'pDC',],

    'Astrocyte':
    ['Astrocyte_A1-like',
    'Astrocyte_reactive',
    'Astrocyte_IFN_response',
    'Astrocyte_homeostatic',
    'Astrocyte_early_activation',],

    'Endothelial':
    ['Endothelial_art',
    'Endothelial_cap',
    'Endothelial_nos',
    'Endothelial_tip',
    'Endothelial_vasc_cycling',
    'Endothelial_vei',],

    'Microglia':
    ['Microglia',
    'Microglia_IFN_response',
    'Microglia_TREM2',
    'Microglia_early_activation',],

    'Macrophage':
    ['Macro_C1Q_MARCOneg',
    'Macro_IFN',
    'Macro_LYVE1',
    'Macro_PLIN2',
    'Macro_cycling'],

    'TILs':
    [ 
 'T_reg',
 'T_CD8_nos',
 'T_CD4_nos',
 'T_other',
 'T_NK_cycling',
 'T_nos',
 'T_IFN_activated',
 'B_nos',
 'Plasma_nos',
 'Plasmablast',
 ]

    }

In [ ]:
for p in range(5,6):
    print(p)
    patient_scores = calculate_patient_neoantigen_scores(
        best_df_mhc,
        percentile_threshold=p, # Peptides with percentile < 1.0 are "good"
        top_n=50                   # Consider top 50 peptides for the third score
    )

    analysis_df = pd.merge(clinical_df, patient_scores, on='patient_id', how='left').dropna()
    sns.scatterplot(analysis_df,x='OS_duration',y='burden_count',hue='OS_status')
    plt.show()
    sns.scatterplot(analysis_df,x='pfs_years',y='burden_count',hue='OS_status')
    plt.show()

In [ ]:
    sns.scatterplot(analysis_df,x='burden_count',y=f'{k}_pct',hue='OS_status');plt.yscale('log')


In [ ]:
f, axs = plt.subplots(2,5,figsize=(15,7))
axs = axs.flat
for k,v in dict_ct_simple.items():
    ax = next(axs)
    ct_pct = (comp_df[v].sum(1)*100).to_frame()
    ct_pct.columns = [f'{k}_pct']
    analysis_df = pd.merge(clinical_df, patient_scores, on='patient_id', how='left').dropna()
    analysis_df = pd.merge(analysis_df, ct_pct, on='patient_id',how='left').dropna()
    sns.scatterplot(analysis_df,x='burden_count',y=f'{k}_pct',hue='OS_status',ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
analysis_df = pd.merge(clinical_df, patient_scores, on='patient_id', how='left').dropna()
analysis_df = pd.merge(analysis_df, t_cell_percentages, on='patient_id',how='left').dropna()
sns.scatterplot(analysis_df,x='burden_count',y='TIL_pct',hue='OS_status');plt.show()
sns.scatterplot(analysis_df,x='OS_duration',y='TIL_pct',hue='OS_status');plt.show()
sns.scatterplot(analysis_df,x='burden_count',y='TIL_pct',hue='OS_status');plt.show()
sns.scatterplot(analysis_df,x='pfs_years',y='TIL_pct',hue='OS_status');plt.show()
sns.boxplot(analysis_df,x='Administrative gender',y='TIL_pct',hue='OS_status');plt.show()
sns.boxplot(analysis_df,x='Administrative gender',y='burden_count',hue='OS_status');plt.show()
sns.boxplot(analysis_df,x='MGMT promoter methylation',y='TIL_pct',hue='OS_status');plt.show()
sns.boxplot(analysis_df,x='MGMT promoter methylation',y='burden_count',hue='OS_status');plt.show()

In [ ]:
# Threshold for a peptide to be considered a "good" neoantigen.
# Here, we select peptides in the top 1.0 percentile of presentation scores.
# PRESENTATION_PERCENTILE_THRESHOLD = 1.0

# # --- 2. Calculate the Patient-Level Score (Neoantigen Burden) ---
# print("Calculating Neoantigen Burden...")
# # Filter for high-quality binders based on the presentation percentile
# analysis_df_filt = analysis_df[analysis_df['presentation_percentile'] < PRESENTATION_PERCENTILE_THRESHOLD]


# ax = plt.subplot()
# analysis_df['presentation_percentile'].hist(ax=ax,bins=100,label='peptide presentation percentile')
# ax.axvline(PRESENTATION_PERCENTILE_THRESHOLD,c='k')
# plt.show()

# # For each patient, count the number of unique high-quality peptides
# # Using nunique('peptide') is important to not double-count a peptide presented by multiple alleles
# neoantigen_burden = analysis_df.groupby('patient_id')['peptide'].nunique().reset_index()
# neoantigen_burden.rename(columns={'peptide': 'neoantigen_burden'}, inplace=True)


# --- 4. Kaplan-Meier Analysis  ---

print("\n--- Running Kaplan-Meier Analysis ---")

median_burden = analysis_df['burden_count'].median()
analysis_df['burden_group'] = 'Low'
analysis_df.loc[analysis_df['burden_count'] > median_burden, 'burden_group'] = 'High'

kmf = KaplanMeierFitter()
ax = plt.subplot(111)

for group_name, group_df in analysis_df.groupby('burden_group'):
    # USE THE NEW COLUMNS HERE
    kmf.fit(durations=group_df['OS_duration'], 
            event_observed=group_df['OS_status'], 
            label=f'{group_name} Burden ({len(group_df)})')
    kmf.plot_survival_function(ax=ax)

low_burden_group = analysis_df[analysis_df['burden_group'] == 'Low']
high_burden_group = analysis_df[analysis_df['burden_group'] == 'High']

# AND HERE
results = logrank_test(
    durations_A=low_burden_group['OS_duration'],
    durations_B=high_burden_group['OS_duration'],
    event_observed_A=low_burden_group['OS_status'],
    event_observed_B=high_burden_group['OS_status']
)

plt.title("Survival by Neoantigen Burden")
plt.xlabel("Time (Overall Survival in years)")
plt.ylabel("Survival Probability")
plt.text(0.1, 0.1, f"Log-rank test p-value: {results.p_value:.4f}", transform=ax.transAxes)
plt.grid(True)
plt.show()

print("\nLog-rank test results:")
results.print_summary()


# --- 5. Cox Proportional Hazards (CPH) Model ---
print("\n--- Running Cox Proportional Hazards Analysis ---")
# USE THE NEW COLUMNS HERE
cph_df = analysis_df[['OS_duration', 'OS_status', 'burden_count']]
cph_df = cph_df.dropna()

print("Crosstab of Burden Group vs. Event Status:")
print(pd.crosstab(analysis_df['burden_group'], analysis_df['OS_status']))


cph = CoxPHFitter()
# AND HERE
cph.fit(cph_df, duration_col='OS_duration', event_col='OS_status')

print("\nCPH Model Summary:")
cph.print_summary()